# Exercise 6 — Build the experiment yourself
Use SVMs, build a voting ensemble, and predict whether an e-bike can finish a trip using features you design.

## When to use AI agents

- **Tasks 1–4: work without AI agents.** Write and understand the code yourself. Use the supplied starter code, API hints and teacher help.
- **Task 5a: follow the teacher demonstration.** The demonstration code is supplied.
- **Task 5b: use your coding agent.** Have it extend **your own completed Task 4 pipeline**, implement the comparison, run experiments and generate the report.
- **Your review is your own work.** Check the agent's output and write `REVIEW.md` yourself; do not ask the agent to write it.

| Task | Your output |
| --- | --- |
| 1. SVM: C and kernels | Comparison table and decision boundaries |
| 2. A tree on paper | Cuts, counts and a small tree |
| 3. Your own bagging | Bootstrap loop and majority vote |
| 4. E-bike features and random forest | Your pipeline and controlled feature experiment |
| 5. Search and agent extension | Start the compulsory hand-in |
| Questions and debugging | Reserve distributed across the lab |

**Write the experiments yourself.** Data loading and plotting are supplied. Use the API hints when needed; complete solutions are in the teacher version only. Start Task 5 in class and finish at home, within **48 hours after your lab**.

<p style="color: #c00000;"><strong>! Compulsory hand-in: complete and submit Task 5b within 48 hours after your lab. Attendance will not be credited unless you complete this requirement.</strong></p>

In [ ]:
%matplotlib inline
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler, FunctionTransformer, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score

DATA = "https://raw.githubusercontent.com/tomasvicar/MLR-public/master/exercises/data/"

## Task 1 — C and kernels
An SVM balances margin size against violations; **smaller C gives stronger regularization**. In the dual formulation, dot products can be replaced by a kernel, allowing curved decision boundaries. Today we measure the effect rather than rederive the optimization problem.

### 1a. Leaves: write a C experiment
Use a `Pipeline` with `StandardScaler` and linear `SVC`. Fit three values of C: 0.01, 1, 100. Collect training accuracy, validation accuracy and the number of support vectors in a table; keep the fitted pipelines in `leaf_models` keyed by C.

**API hints:** `Pipeline([("scale", ...), ("svm", ...)])`, `.fit(X, y)`, `.score(X, y)`, `.named_steps["svm"].n_support_.sum()`. Fit the scaler through the pipeline on training data only.

In [ ]:
leaves = pd.read_csv(DATA + "leaves.csv")
pair = leaves[leaves["Tree type"].isin(["maple", "cherry"])]
X_leaf = pair[["Leaf height", "Leaf width"]].to_numpy()
y_leaf = (pair["Tree type"].to_numpy() == "cherry").astype(int)
X_leaf_train, X_leaf_val, y_leaf_train, y_leaf_val = train_test_split(
    X_leaf, y_leaf, test_size=0.3, stratify=y_leaf, random_state=42)
print("Training / validation leaves:", len(y_leaf_train), len(y_leaf_val))

In [ ]:
# 1. Loop over C = 0.01, 1, 100; build a StandardScaler + linear SVC pipeline.
# 2. Fit on the training leaves; store each fitted model in leaf_models[C].
# 3. Collect training/validation accuracy and support-vector counts in rows.
# 4. Display the rows as a table named leaf_results.
# TODO: your code here
...

In [ ]:
#@title Supplied helper: boundaries, margins and support vectors in original coordinates
def plot_boundary(model, X, y, ax, title=""):
    """Plot a fitted scale/SVC pipeline; X is the original training array."""
    span = np.maximum(np.ptp(X, axis=0), 1)
    low, high = X.min(axis=0) - 0.1 * span, X.max(axis=0) + 0.1 * span
    xx, yy = np.meshgrid(np.linspace(low[0], high[0], 140),
                         np.linspace(low[1], high[1], 140))
    grid = np.column_stack([xx.ravel(), yy.ravel()])
    ax.contourf(xx, yy, model.predict(grid).reshape(xx.shape),
                levels=[-0.5, 0.5, 1.5], cmap="coolwarm", alpha=0.2)
    ax.contour(xx, yy, model.decision_function(grid).reshape(xx.shape),
               levels=[-1, 0, 1], colors="black", linestyles=["--", "-", "--"], linewidths=0.7)
    ax.scatter(X[:, 0], X[:, 1], c=y, cmap="coolwarm", s=16)
    support = X[model.named_steps["svm"].support_]
    ax.scatter(support[:, 0], support[:, 1], s=55, facecolors="none", edgecolors="black")
    ax.set_title(title)

fig, axes = plt.subplots(1, 3, figsize=(13, 3.5))
for ax, (C, model) in zip(axes, leaf_models.items()):
    plot_boundary(model, X_leaf_train, y_leaf_train, ax, f"C = {C}")
    ax.set_xlabel("Height [cm]")
    ax.set_ylabel("Width [cm]")
plt.tight_layout()
plt.show()

### 1b. Circles: choose a kernel
On the supplied split, construct and compare three scaling + SVM pipelines: linear, RBF with gamma=1, RBF with gamma=1000; keep C=1. Write the loop and a train/validation score table. Use `plot_boundary` to inspect all three models.

**Explain:** Which boundary follows the circles? Does near-perfect training accuracy imply good validation accuracy? Explain the actual measurements; do not assume accuracy or support-vector counts must be monotonic.

*Optional:* add a polynomial kernel with degree=3 and coef0=1.

In [ ]:
from sklearn.datasets import make_circles
X_circle, y_circle = make_circles(n_samples=300, factor=0.4, noise=0.12, random_state=0)
X_circle_train, X_circle_val, y_circle_train, y_circle_val = train_test_split(
    X_circle, y_circle, test_size=0.3, stratify=y_circle, random_state=0)

In [ ]:
# 1. Compare linear SVM, RBF gamma=1 and RBF gamma=1000, all with C=1.
# 2. Scale inside each pipeline; fit only on the circle training split.
# 3. Record training/validation accuracy and plot each model with plot_boundary.
# 4. Display circle_results and the three boundary plots.
# TODO: your code here
...

Your interpretation of C and the kernels (two or three sentences):

## Task 2 — Draw a tree and count correct predictions
**Laptops closed.** A decision tree divides the plane with one horizontal or vertical cut at a time. For this exercise, choose cuts by accuracy.

| Point | x₁ | x₂ | Class |
| --- | --- | --- | --- |
| 1 | 1 | 1 | A |
| 2 | 2 | 3 | A |
| 3 | 3 | 2 | B |
| 4 | 4 | 4 | B |
| 5 | 5 | 1 | A |
| 6 | 6 | 3 | B |

1. Draw the six points using two symbols or colours.
2. Try **x₁ ≤ 1.5**, **x₁ ≤ 2.5**, **x₂ ≤ 2.5**. Assign each side its majority class.
3. Count correct predictions across **all six points** and choose the best candidate.
4. Split its remaining impure region once more. Draw the resulting tree and trace point 5 through it.

A tied majority can use either label: the accuracy is unchanged. These are candidate cuts, not an exhaustive threshold search. `DecisionTreeClassifier` normally uses Gini (or entropy/log loss), **not this accuracy rule**.

## Task 3 — Build your own bagging
Several trees can disagree because each sees a different bootstrap sample. Test whether voting helps on the **same leaf split as Task 1**.

The code below already trains a reference tree on all training rows and **one tree on a bootstrap sample**. Run it and inspect the number of unique sampled indices.

1. Read the supplied example: it draws n indices **with replacement** and uses the same indices for X and y.
2. Extend this example into a loop that trains **11 trees**, using random_state=0,...,10. Draw a fresh bootstrap sample inside each iteration; initialize the random generator once before the loop.
3. Store each fitted tree in a list (`trees`) and its number of unique training rows in `unique_counts`.
4. Form a validation prediction matrix with one column per tree.
5. Implement majority voting: predict 1 if more than half the trees predict 1.
6. Display individual-tree accuracies, single-tree accuracy and ensemble accuracy. Did voting help this time?

**API hints:** `np.random.default_rng(0)`, `rng.choice(n, size=n, replace=True)`, `np.column_stack(...)`, `.mean(axis=1)`. No `BaggingClassifier` shortcut. No scaling is needed here. Eleven binary votes cannot tie.

**Check your implementation:** the prediction matrix must have one row per validation leaf and 11 columns. Every bootstrap sample must contain exactly the training-set size in paired rows, although some indices repeat. Trace one row of 11 votes by hand and compare with your ensemble prediction.

In [ ]:
# Reference: one tree trained on all training rows.
single_tree = DecisionTreeClassifier(random_state=0).fit(X_leaf_train, y_leaf_train)

# Starting example: one tree trained on a bootstrap sample.
rng = np.random.default_rng(0)
n_train = len(y_leaf_train)
indices = rng.choice(n_train, size=n_train, replace=True)
tree = DecisionTreeClassifier(random_state=0)
tree.fit(X_leaf_train[indices], y_leaf_train[indices])
print("Sampled rows:", len(indices), "; unique rows:", len(np.unique(indices)))
print("Bootstrap tree validation accuracy:", tree.score(X_leaf_val, y_leaf_val))

In [ ]:
# 1. Initialize rng once and create empty trees and unique_counts lists.
# 2. Repeat the supplied bootstrap-and-fit code 11 times; save each tree/count.
# 3. Stack validation predictions into votes (one column per tree).
# 4. Compute bagging_prediction by majority vote; compare individual/ensemble accuracy.
# TODO: your code here
...

Why do the trees disagree, and what did their vote change?

## Task 4 — Will the e-bike finish the trip?
You operate an e-bike service. Before departure, predict **runs_out=1** if the bike will run out of charge before reaching its planned destination.

These are **synthetic teaching data**, not measurements or a validated battery model. Trips are independent simulations. Some measurements are missing, some extremes are legitimate, and a few readings are impossible. Inputs are available before departure; the outcome is known afterwards.

| Columns | Meaning / units |
| --- | --- |
| capacity_wh, charge_pct, battery_age_years | Nominal battery energy [Wh], charge [%], age [years] |
| distance_km, ascent_m | Planned route length [km], total climb [m] |
| temperature_c, headwind_kmh, payload_kg | Conditions [°C, km/h, kg]; headwind is nonnegative in this simulation |
| surface | paved / mixed / gravel |
| bike_type | city / touring / cargo |
| assistance | eco / normal / turbo |
| runs_out | Binary outcome; never a feature |

A `RandomForestClassifier` adds random feature selection at each split to the bootstrap idea you just implemented. Use **100 trees, random_state=0**, with other model settings unchanged throughout Task 4.

### 4a. Inspect and prepare
Inspect **training data only** with `.isna().sum()`, `.describe()` and category counts. Identify a legitimate extreme and an impossible value. Keep valid extremes: an unusual trip is not automatically a mistake.

Write `clean_readings(frame)`: copy the table; replace charge outside [0,100] and distance ≤0 with NaN. These fixed physical rules learn nothing from the sample. Other provided numerical extremes are valid for this exercise.

Then assemble a pipeline with:
- cleaning via `FunctionTransformer(clean_readings)`;
- `ColumnTransformer`: numeric median imputation; categorical constant `"missing"` imputation then one-hot encoding with unknown categories ignored;
- `RandomForestClassifier`.

**API hints:** `SimpleImputer(strategy="median").set_output(transform="pandas")` retains numeric column names for the next task. Put it in a numeric `Pipeline` with steps `impute` and `features`; initially `features` is `FunctionTransformer()` (identity). Use `OneHotEncoder(handle_unknown="ignore", sparse_output=False)`. ColumnTransformer entries are `(name, transformer, columns)`. Name its branches `num`, `cat`, and the outer steps `clean`, `prep`, `model`.

The whole pipeline goes into CV: never impute the complete dataset beforehand. The forest needs no scaling.

**Private preview:** download this notebook and `ex06_ebikes.csv` from the private course page, then upload both to Colab (CSV in the Files sidebar). The loader uses this local CSV first. The public raw-data URL becomes available only after public release.

In [ ]:
from pathlib import Path
# For the private preview, upload the downloaded CSV next to this notebook.
local_csv = Path("ex06_ebikes.csv")
ebikes = pd.read_csv(local_csv if local_csv.exists() else DATA + "ex06_ebikes.csv")
NUMERIC = ["capacity_wh", "charge_pct", "battery_age_years", "distance_km",
           "ascent_m", "temperature_c", "headwind_kmh", "payload_kg"]
CATEGORICAL = ["surface", "bike_type", "assistance"]
X = ebikes[NUMERIC + CATEGORICAL]
y = ebikes["runs_out"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=42)
CV = StratifiedKFold(n_splits=3, shuffle=True, random_state=0)
print("Training trips:", len(y_train), "; held-out test trips:", len(y_test))
# Keep X_test and y_test closed until final evaluation in the hand-in.

In [ ]:
# 1. Inspect missing values, numeric summaries and categories in X_train only.
# 2. Write clean_readings: copy the input; replace invalid charge/distance with NaN.
# 3. Build numeric median imputation + identity features and categorical imputation + encoding.
# 4. Combine clean, prep and a 100-tree forest (random_state=0) into baseline.
# TODO: your code here
...

### 4b. Establish the baseline
The starter code below evaluates your **whole pipeline** with `cross_val_score(..., X_train, y_train, cv=CV, scoring="accuracy")`. It saves the three scores as `baseline_scores`. Extend the example to evaluate `DummyClassifier(strategy="most_frequent")` on exactly these folds. Report mean and fold standard deviation (`std(ddof=0)`), and the majority-class score. Do not inspect test accuracy.

In [ ]:
# Starting example: evaluate the whole forest pipeline on training folds only.
# cross_val_score refits preprocessing inside each fold, preventing leakage.
scoring = "accuracy"
baseline_scores = cross_val_score(
    baseline, X_train, y_train, cv=CV, scoring=scoring)
print("Forest: mean", baseline_scores.mean(), "fold SD", baseline_scores.std(ddof=0))

# Majority-class reference: this model always predicts the most common training label.
dummy = DummyClassifier(strategy="most_frequent")
# 1. Repeat the cross_val_score call above with dummy; save it as dummy_scores.
# 2. Keep X_train, y_train, CV and scoring unchanged for a fair comparison.
# 3. Print its mean and fold SD (ddof=0); compare with the forest. Keep the test set closed.
# TODO: your code here
...

### 4c. Create and test features
Start with **available_wh = capacity_wh × charge_pct / 100**. Write `add_energy(frame)` to return a copy with this column appended. It receives numeric data **after median imputation within each training fold**.

Clone your baseline and replace its numeric feature function:
`clone(baseline).set_params(prep__num__features__func=add_energy)`.
Evaluate it on the same `CV` and unchanged forest settings.

Then choose **one further idea**, write down your hypothesis first, and compare it with the first addition:

| Idea | Meaning |
| --- | --- |
| wh_per_km = available_wh / distance_km | Energy budget for the planned distance |
| ascent_per_km = ascent_m / distance_km | Route intensity |
| Cold-weather deficit | Degrees below a chosen reference temperature |
| Numeric × category condition | For example, a long ride in turbo mode; requires adjusting the transformer to access the category |
| Missing-measurement flag | Must be recorded before imputation; no promise that missingness is informative |

**Core work:** original features → energy product → one chosen extension. The teacher example uses energy budget and climb per km. Ratios are computed after invalid distances become missing and fold-fitted imputation replaces them with positive distances. Do not copy the simulator equation or use any post-trip outcome as a feature.

Name your final extended pipeline `engineered`; the supplied Task 5 demonstration uses this variable.

**Check:** on an imputed row with capacity=500 Wh, charge=60%, distance=20 km and ascent=400 m, calculate your derived features by hand and compare with your function. Confirm that it retains the original columns and does not modify its input.

Keep a **three-row result table** with mean CV accuracy and fold SD. A feature need not improve accuracy. Explain what your result supports; the fold SD is context, not a significance test.

*Optional discussion:* ratios and differences; sin/cos for periodic inputs in other datasets; why log transforms often matter more for distance-based models than for trees.

In [ ]:
# 1. Write add_energy: copy the frame and append capacity_wh * charge_pct / 100.
# 2. Clone baseline and set prep__num__features__func=add_energy.
# 3. Evaluate on X_train/y_train with the same CV; save energy_scores.
# 4. Print the mean accuracy and fold standard deviation.
# TODO: your code here
...

My next feature and hypothesis (write before running it):

In [ ]:
# 1. Start your feature function with result = add_energy(frame), then add your extra feature.
#    Keep available_wh and the original columns; do not modify the input frame.
# 2. Clone baseline and set its numeric feature function; name the pipeline engineered.
# 3. Evaluate with the same CV and forest settings; save engineered_scores.
# 4. Display a three-row table: baseline, energy feature, your extension (mean and SD).
# 5. Check your feature function on the hand-computed example from the instructions.
# TODO: your code here
...

Which feature helped, or failed to help? What did you hold fixed?

## Task 5 — Search and extend with an agent
### 5a. Teacher demonstration
A grid is a loop around the **same CV evaluation**. We vary minimum leaf size and feature subsampling on the same e-bike pipeline. All fitting, including imputation, remains inside CV; the test set stays closed.

The following cells are supplied demonstration code. If your own feature experiment is unfinished, follow the projector and finish it before asking the agent to extend it.

In [ ]:
# Teacher demonstration: four configurations, each measured on the same folds.
grid_rows = []
for min_samples_leaf in [1, 5]:
    for max_features in ["sqrt", 1.0]:
        candidate = clone(engineered).set_params(
            model__min_samples_leaf=min_samples_leaf, model__max_features=max_features)
        scores = cross_val_score(candidate, X_train, y_train, cv=CV, scoring="accuracy")
        grid_rows.append({"min_samples_leaf": min_samples_leaf, "max_features": max_features,
                          "cv_mean": scores.mean(), "cv_sd": scores.std()})
grid_results = pd.DataFrame(grid_rows).sort_values("cv_mean", ascending=False)
display(grid_results)

Optuna proposes parameters through a `trial`; the objective returns the same mean CV accuracy. A seeded TPE sampler uses earlier trials to guide later proposals after its initial random trials. This six-trial demonstration is deliberately small; it does not prove an advantage over a grid. The highest score among many attempts is an optimistic estimate.

In [ ]:
import importlib.util
import subprocess
import sys
if importlib.util.find_spec("optuna") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "optuna"], check=True)
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

def objective(trial):
    candidate = clone(engineered).set_params(
        model__min_samples_leaf=trial.suggest_int("min_samples_leaf", 1, 12),
        model__max_features=trial.suggest_categorical("max_features", ["sqrt", 0.5, 1.0]))
    scores = cross_val_score(candidate, X_train, y_train, cv=CV, scoring="accuracy")
    trial.set_user_attr("cv_sd", float(scores.std()))
    return float(scores.mean())

study = optuna.create_study(direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=0, n_startup_trials=3))
study.optimize(objective, n_trials=6, n_jobs=1)
print("Best CV:", study.best_value, "; parameters:", study.best_params)
plt.plot(np.arange(1, 7), np.maximum.accumulate([t.value for t in study.trials]), "o-")
plt.axhline(grid_results.cv_mean.max(), linestyle="--", label="Grid best")
plt.xlabel("Completed trials")
plt.ylabel("Best-so-far CV accuracy")
plt.legend()
plt.show()

### 5b. Start your compulsory agent extension

<p style="color: #c00000;"><strong>! Compulsory hand-in: complete and submit Task 5b within 48 hours after your lab. Attendance will not be credited unless you complete this requirement.</strong></p>

The agent extends **your own e-bike experiment**. You review the result. Use your laptop terminal and the coding agent from the fifth lab. Download this notebook as `ex06_notebook.ipynb`; create a new **private** repository `mlr-ex06`. Your public repository from the fifth lab is not a suitable hand-in location.

**Optional editor:** if you want to use VS Code on Windows, install it from PowerShell or Windows Terminal:

```powershell
winget install --id Microsoft.VisualStudioCode --exact --scope user --accept-package-agreements --accept-source-agreements
```

Share `mlr-ex06` with **MLRteacher**: repository **Settings → Collaborators → Add people → MLRteacher**. A pending invitation is sufficient. Submit by pushing within **48 hours after the end of your lab**. Hand-ins are reviewed automatically by AI.

If GitHub CLI (`gh`) is installed and authenticated (`gh auth status`), you can simply ask your coding agent:

> Use `gh` to invite `MLRteacher` as a collaborator to my private `mlr-ex06` repository. Check the repository owner and confirm that the invitation is pending or access already exists.

Your signed-in GitHub account must have permission to manage collaborators for that repository.

<p style="color: #c00000;"><strong>! Plagiarism check: all hand-ins will be checked for plagiarism.</strong></p>

If your notebook still contains TODOs in the e-bike pipeline, finish them first or ask the teacher for help. Do not replace your work with a downloaded completed module. Without a working agent, implement the same specification yourself at home.

**Paste this specification to your agent:**

> Extend my `ex06_notebook.ipynb` e-bike experiment. First inspect my code. If required functions or experiments are unfinished, tell me what is missing before proceeding.
>
> 1. Copy my data loading, fixed cleaning rules, feature functions and baseline into `common.py`, with the required imports. Preserve their behavior. Use `ex06_ebikes.csv`, `runs_out` as target, test_size=0.25, stratify=y, random_state=42. Select all features and models using only the training subset and `StratifiedKFold(n_splits=3, shuffle=True, random_state=0)`. Imputation and scaling must be fitted inside each fold. Retain the untouched test subset for final reporting.
> 2. Implement `bayes_compare.py --trials 20`: compare k-NN, SVM, logistic regression, random forest (100 trees), and `HistGradientBoostingClassifier`, using Optuna with the **same number of trials per family**. Trial parameters must be meaningful for the model. Use log scales for C, gamma, learning rate and positive regularization strengths; suggest gamma only for RBF SVM. Tune sensible parameters such as neighbors/weights, C/kernel, minimum leaf size/feature subsampling, or learning rate/leaf count. Do not add a tree-depth experiment. Bound parameters for the sample size.
> 3. Limit preparation choices to numeric mean/median imputation and original/engineered features; preserve the physically motivated cleaning and categorical missing-label + one-hot encoding. Keep my chosen engineered feature set. Use StandardScaler for k-NN, SVM and logistic regression, after numeric feature creation; do not search scalers for trees. Fix the forest's random_state=0 and all other model, sampler and split seeds. Run Optuna sequentially (`n_jobs=1`) for reproducibility.
> 4. Record each trial in `trials.csv` (family, trial number, applied parameters, CV mean and fold SD). Choose the best configuration per family **by CV**, then fix all choices before evaluating on the test subset. Produce `results.csv`: best parameters, best trial number, CV mean and fold SD, final test accuracy. Include my unchanged original-feature forest and the majority-class classifier as reference rows, evaluated with the same folds. Check that the baseline agrees with my notebook. Select the overall winner by CV, not test scores.
> 5. Produce `convergence.png` (best-so-far CV accuracy versus completed trials for each family) and a self-contained `report.html` containing the plot, table, baseline comparison, trial budget, split and seeds, package versions and a cautious conclusion. Fold SD is not a significance test. Report limitations and validation-selection optimism; do not promise a real battery-safety predictor.
> 6. Add `pyproject.toml` and `uv.lock`, a README with commands from a fresh clone and measured runtime. Implement `--cv-only` to skip all test prediction and scoring. Run a small smoke test (`--trials 2 --cv-only`) first, then the full `--trials 20` comparison twice; verify identical `results.csv`. Save my notebook too. Do not write `REVIEW.md`: that is my own work.

**Required files:** `common.py`, `bayes_compare.py`, `ex06_notebook.ipynb`, dependency files, `README.md`, `trials.csv`, `results.csv`, `convergence.png`, `report.html`, and your own `REVIEW.md`. Include any additional modules or data required to reproduce the run. Finish the full run and review at home; no particular laptop runtime is promised.

**Your eight-point review — write evidence in `REVIEW.md`, yourself:**

1. **Preprocessing inside CV:** locate the full pipeline in the objective; check that imputation and scaling are never fitted on validation/test rows. Confirm the fixed cleaning bounds and safe ratios.
2. **Test isolation:** find every use of X_test/y_test. Were all model and feature choices fixed before these scores were calculated? No post-trip variables or target-derived features may enter X.
3. **Fair comparison:** count completed trials, compare folds and data rows across algorithms, and verify the two reference rows. Equal trial counts do not imply equal compute effort.
4. **Search ranges:** inspect logarithmic versus linear ranges, sensible neighbor/leaf bounds and the limited feature/preparation choices. Are all returned parameters applied?
5. **Conditional parameters:** gamma only for RBF; scaling for the distance/linear models, not a meaningless tree scaling search. Check the best parameter dictionaries.
6. **Reproducibility:** inspect every seed and compare the two full-run CSVs. Record the command and evidence; do not include runtime in a table expected to be byte-identical.
7. **Agreement with your work:** compare the original-feature forest with Task 4 and explain any discrepancy. Look at convergence and whether extra features helped. Large gains deserve investigation, not automatic acceptance.
8. **Conclusions:** distinguish validation selection from final testing. Compare differences with fold variability without treating it as a formal significance test. State what synthetic data cannot establish about real e-bikes.

For each point record the location (file/line or notebook cell), what you checked, your finding, and how any problem was fixed. “No problem found” needs evidence too. The agent writes the code and generated report; **you write the review**.

## Summary
- C controls SVM regularization; kernels change the decision boundary.
- A tree makes successive cuts. Bagging trains trees on bootstrap samples and votes; random forests also randomize split-feature candidates.
- Physical bounds distinguish impossible readings from valid extremes. Learned preprocessing belongs inside validation.
- Domain features combine quantities the model would otherwise have to relate itself. Compare them with the same model and folds.
- Search and agents extend an experiment; you remain responsible for isolation of test data, reproducibility and justified conclusions.